# Pillar3k-small — distill pillar3k into a 4× smaller student (10b × 128ch)

Compress the deployed best **pillar3k** (10b × 256ch, 11.9M params) into a **10b × 128ch** student (3.0M params) for faster generation + browser deploy.

**Premise (benchmarked on M5):** 128ch = **3.96× fewer params**; **~2.4× faster** in FLOP-bound regimes (CPU batch=1 / browser = 2.43×; large-batch eval = 2.64×), 1.4–1.8× at MPS MCTS batch sizes. So: *4× smaller, ~2.4× faster* (not 4× faster — stem/heads/BN don't scale with channels²).

**Method — direct teacher-policy distillation.** `distill_pillar3k.pt` = 3,846,619 states (selfplay broad-normal + all crisis escapes) whose policy targets ARE **pillar3k's top-5 legal-move policy** (relabeled by `distill_relabel.py`). The student trains to MATCH pillar3k:
- **`--channels 128`** (the only arch change; train_path_b supports it).
- **`--target-temperature 0.5`** — SHARPEN pillar3k's targets onto the argmax. A greedy student must nail the *top move*; a faithful soft match (T=1.0) learned far too slowly (early-death spirals, mean ~800 at ep11) because pillar3k's policy is soft (top-share 0.34) and the argmax signal was weak. T=0.5 commits the student to pillar3k's top move fast.
- **NO `--decisiveness-power`** — that was for fine-tuning a peaked base; here we want the student to learn pillar3k everywhere (normal + escapes).
- **From scratch at `--lr 1e-3 --warmup-epochs 3`** — the *proven* from-scratch recipe (HISTORY: pillar2z/V12). The first runs used lr=3e-4 (the warm-start *fine-tuning* LR) and crawled: argmax-match to pillar3k was only 12.6%→15.6% over ep8→ep11 (~1%/epoch, diagnosed via `diag_student_match.py`). From-scratch needs the 3× higher LR. (No warm-start possible — arch differs from any 256ch checkpoint.)

**Eval = degradation vs the teacher.** Bar = pillar3k (5k, 775000–779999): **mean 43,390 / P50 31,016 / P10 5,010 / %<1000 1.3%**. A few-% drop is a great trade for 4× smaller + 2.4× faster. Val here is a faithful CE to pillar3k's policy and SHOULD fall monotonically (unlike the decisiveness-weighted runs) — but still pick the epoch by gameplay floor.

**Upload to Drive (`MyDrive/alphatrain/`):** `colorlines_pillar3d_v2.tar.gz` (already there) + `distill_pillar3k.pt.gz` (346 MB).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time
DRIVE='/content/drive/MyDrive/alphatrain'
!cp {DRIVE}/colorlines_pillar3d_v2.tar.gz /content/
!cd /content && tar xzf colorlines_pillar3d_v2.tar.gz
os.makedirs('/content/alphatrain/data', exist_ok=True)
t0=time.time()
!cp {DRIVE}/distill_pillar3k.pt.gz /content/distill_pillar3k.pt.gz
gz=os.path.getsize('/content/distill_pillar3k.pt.gz'); print(f'.gz: {gz:,} bytes')
assert gz == 346_146_261, f'.gz truncated! got {gz}; re-upload distill_pillar3k.pt.gz'
!gunzip -t /content/distill_pillar3k.pt.gz && echo '.gz integrity OK'
!gzip -dc /content/distill_pillar3k.pt.gz > /content/alphatrain/data/distill_pillar3k.pt
pt=os.path.getsize('/content/alphatrain/data/distill_pillar3k.pt')
assert pt == 1_473_260_729, f'.pt size wrong! got {pt}'
print(f'corpus: {pt/1e9:.2f} GB, 3,846,619 states ({time.time()-t0:.0f}s)')
!rm /content/distill_pillar3k.pt.gz
!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g=torch.cuda.get_device_properties(0); print(f'GPU {torch.cuda.get_device_name(0)} | {g.total_memory/1e9:.0f} GB')

In [ ]:
# ===== CONFIG =====
CHANNELS = 128            # 4x smaller student (vs teacher's 256). Fallback: 192 if degradation too big.
EPOCHS   = 40             # from-scratch 4x-smaller student -> needs the most epochs
RUN      = "pillar3k_small128_lr1e3"
print(f'RUN={RUN}  CHANNELS={CHANNELS}  EPOCHS={EPOCHS}  (distill pillar3k, T=0.5 sharpen-argmax, from scratch)')

In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/distill_pillar3k.pt \
    --channels {CHANNELS} --amp --compile \
    --epochs {EPOCHS} --batch-size 32768 --lr 1e-3 --warmup-epochs 3 \
    --target-temperature 0.5 \
    --copy-to /content/drive/MyDrive/alphatrain/{RUN}_best.pt \
    --save-dir /content/checkpoints/{RUN} 2>&1 | tee /content/{RUN}_train.log
# val SHOULD fall monotonically here (faithful CE to pillar3k's policy).

In [ ]:
import shutil, os, glob
DRIVE='/content/drive/MyDrive/alphatrain'
for f in sorted(glob.glob(f'/content/checkpoints/{RUN}/epoch_*.pt')):
    dst=f'{DRIVE}/{RUN}_{os.path.basename(f)}'; shutil.copy(f,dst); print('Saved', dst)
for f in ['best.pt','latest.pt']:
    s=f'/content/checkpoints/{RUN}/{f}'
    if os.path.exists(s): shutil.copy(s,f'{DRIVE}/{RUN}_{f}'); print('Saved', f'{DRIVE}/{RUN}_{f}')

## Eval — degradation vs the teacher (pillar3k)

On M5, per epoch (pick by gameplay floor):
```
python -m scripts.eval_policy --model <student_ckpt> --device cuda --batch 1024 \
    --seed-start 775000 --seed-end 779999    # full 5k
```
**Bar = pillar3k (5k): mean 43,390 / P50 31,016 / P10 5,010 / %<1000 1.3%.**
- **Within a few % on mean AND floor** → success: 4× smaller / ~2.4× faster for a small score cost. Use it as the fast generation engine (re-time crisis mining) and the browser candidate.
- **Large floor drop (P10, %<1000)** → the 128ch capacity can't hold pillar3k's escapes. Fallbacks: (a) relabel pillar3k's OWN self-play states (exact deployment distribution, cleaner than pillar3f's selfplay) and retrain; (b) step up to 6b×192ch (≈2× smaller, less aggressive); (c) longer training.

**Speed is already characterized** (M5): 128ch is ~2.4× faster CPU/batch=1 and large-batch, 1.4–1.8× at MCTS batch sizes. Re-benchmark crisis_mining throughput with the student as the MCTS prior to confirm the generation speedup in practice.